<a href="https://colab.research.google.com/github/aliaghababaee/SLO-and-OCT/blob/main/AlexNet%20with%20PyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
import torch.utils.data as data
import torchvision
import torchvision.transforms.functional as TF
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import copy
from sklearn.model_selection import StratifiedGroupKFold

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
csv_data = pd.read_csv ('/content/drive/MyDrive/projection images iran/total_slo_newer.csv')

In [ ]:
class SLO_Dataset (data.Dataset):
  def __init__ (self, indices, transform, csv = csv_data, root_dir = '/content/drive/MyDrive/projection images iran/SLO_new'):
    super ().__init__()
    self.transform = transform
    self.csv = csv_data
    self.root_dir = root_dir
    self.indices = indices

  def __len__ (self):
    return len (self.indices)

  def __getitem__ (self, index):

    img_name = os.path.join (self.root_dir, self.csv.iloc [self.indices [index], 1])
    img = Image.open (img_name)

    #converting od to os:
    #each filename consists of three parts sperated by "."; for instance, in img_files [0] the three parts are as follows:
    #part1: "hc14_spectralis_macula_v1_s1_R", part2: ".OD", part3:".png"
    part1_part2, part3 = os.path.splitext (self.csv.iloc [self.indices [index], 1])
    part1, part2 = os.path.splitext (part1_part2)

    if part2 == '.OD':
      img = img.transpose (Image.FLIP_LEFT_RIGHT)

    if self.transform:
      img = self.transform (img)

    img = torch.tensor (np.array (img) / 256, dtype = torch.float32)
    img = img.permute ((2, 0, 1))
    label = torch.tensor (self.csv.iloc [self.indices [index], 0])

    return img, label

In [ ]:
train_transform = transforms.Compose ([transforms.Resize (128),
                                       transforms.ColorJitter (brightness = [0.2, 1.5]),
                                       #transforms.RandomRotation (5),
                                       transforms.RandomVerticalFlip (),
                                       transforms.RandomApply([transforms.Lambda(lambda x: TF.affine(x, angle = 5,
                                                                                                     scale = 1,
                                                                                                     shear = 0,
                                                                                                     translate=(30, 5)))], p= 0.5),
                                       #transforms.RandomApply (transforms.Compose ([transforms.CenterCrop (),
                                                                                    #transforms.Resize ()]),
                                                              # p = 0.5)

                                       #transforms.CenterCrop ()
                                       #transforms.RandomAffine (degrees = 5, scale= (0.5, 0.6), translate = (0.1, 0.1)),
                                       transforms.Grayscale (num_output_channels = 3)])
test_transform = transforms.Compose ([transforms.Resize (128),
                                      transforms.Grayscale (num_output_channels = 3)])


#mean and SD

In [ ]:
class EarlyStopper:
    def __init__(self, patience=1, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.min_validation_loss = np.inf

    def early_stop(self, validation_loss):
        if validation_loss < self.min_validation_loss:
            self.min_validation_loss = validation_loss
            self.counter = 0
        elif validation_loss > (self.min_validation_loss + self.min_delta):
            self.counter += 1
            if self.counter >= self.patience:
                return True
        return False

In [ ]:
alexnet = torchvision.models.alexnet (pretrained=True)

In [ ]:
torch.save(alexnet.state_dict(), '/content/drive/MyDrive/projection images iran/Models/alexnet.pth')

In [ ]:
alexnet = torchvision.models.alexnet(pretrained=False)  # Create a new instance of AlexNet

In [ ]:
alexnet.load_state_dict(torch.load('/content/drive/MyDrive/projection images iran/Models/alexnet.pth'))

In [ ]:
for param in alexnet.parameters ():
  param.requires_grad = False

In [ ]:
device = torch.device (('cuda') if torch.cuda.is_available () else torch.device ('cpu'))

In [ ]:
def train_alexnet (model, device, train_dataloader, test_dataloader, num_epochs, loss_fn, optimizer, lr_scheduler, early_stopper):

  best_model_wts = copy.deepcopy (model.state_dict ())
  best_acc = 0
  best_epoch = 0

  train_losses = []
  test_losses = []
  train_accs = []
  test_accs = []


  for epoch in range (num_epochs):

    model.train ()

    train_loss = 0.0

    num_correct = 0

    for image, label in train_dataloader:
      image, label = image.to (device), label.to (device)
      label = label.float ()

      output = model (image)

      loss = loss_fn (output, label.view (-1, 1))

      train_loss +=  loss.item () * image.size (0)

      optimizer.zero_grad ()

      loss.backward ()

      optimizer.step ()

      pred = output > 0

      num_correct += torch.sum (pred == label.view (-1, 1))

    train_loss_epoch = train_loss / len (train_dataloader.dataset)

    train_losses.append (train_loss_epoch)

    train_acc_epoch = num_correct.double () / len (train_dataloader.dataset)

    train_accs.append (train_acc_epoch.item ())

    #Validation phase

    model.eval ()

    test_loss = 0.0

    num_correct = 0

    with torch.no_grad ():
      for image, label in test_dataloader:
        image, label = image.to (device), label.to (device)
        label = label.float ()
        output = model (image)
        loss = loss_fn (output, label.view (-1, 1))
        test_loss += loss.item () * image.size (0)
        pred = output > 0
        num_correct += torch.sum (pred == label.view (-1, 1))
    test_loss_epoch = test_loss / len (test_dataloader.dataset)
    test_losses.append (test_loss_epoch)
    test_acc_epoch = num_correct.double () / len (test_dataloader.dataset)
    test_accs.append (test_acc_epoch.item ())
    #print (f'test loss in epoch {epoch + 1} / {num_epochs} is {test_acc_epoch}')


    lr_scheduler.step (test_loss_epoch)

    if test_acc_epoch > best_acc:
      best_acc = test_acc_epoch
      best_epoch = epoch
      best_model_wts = copy.deepcopy (model.state_dict ())



    if early_stopper.early_stop (test_loss_epoch):
      print (f'we are at epoch {epoch}')
      break


  model.load_state_dict (best_model_wts)

  return  model, train_losses, test_losses, train_accs, test_accs, best_epoch, best_acc



In [ ]:
def test_alexnet (model, device, test_dataloader, loss_fn):

  model.eval ()
  test_loss = 0.0
  num_correct = 0

  with torch.no_grad ():
    for image, label in test_dataloader:
      image, label = image.to (device), label.to (device)
      label = label.float ()
      output = model (image)
      loss = loss_fn (output, label.view (-1, 1))
      test_loss += loss.item () * image.size (0)
      pred = output > 0
      num_correct += torch.sum (pred == label.view (-1, 1))
  test_loss = test_loss / len (test_dataloader.dataset)
  test_acc = num_correct.double () / len (test_dataloader.dataset)

    #print (f'test loss in epoch {epoch + 1} / {num_epochs} is {test_acc_epoch}')
  #lr_scheduler.step (test_loss)
  return test_loss, test_acc


In [ ]:
n_folds = 5

skfg = StratifiedGroupKFold (n_splits = n_folds, shuffle = True, random_state = None)

labels_total = csv_data.iloc [:, 0]

groups_total = csv_data.iloc [:, 2]

losses_folds_train = []
losses_folds_test = []
accs_folds_train = []
accs_folds_test = []

epoch_folds = []

model_folds = []

best_accs_returned_by_the_training_model = []

best_accs_returned_by_the_test_model = []

acc_folds = 0

for train_indices, test_indices in skfg.split (np.zeros ((len (labels_total))), labels_total, groups_total):

  train_dataset = SLO_Dataset (indices = train_indices, transform = train_transform)

  test_dataset = SLO_Dataset (indices = test_indices, transform = test_transform)


  train_dataloader = data.DataLoader (train_dataset, batch_size = 64, shuffle = True)
  test_dataloader = data.DataLoader (test_dataset, batch_size = 64, shuffle = False)

  loss_fn = nn.BCEWithLogitsLoss ()

  alexnet.classifier =  nn.Sequential (nn.Dropout (0.5),
                                     nn.Linear (256 * 36, 32 * 36),
                                     nn.ReLU (),
                                     nn.Dropout (0.6),
                                     #nn.Linear (4096, 4096),
                                     #nn.ReLU (),
                                     #nn.Linear (4096, 1000),
                                     #nn.ReLU (),
                                     #nn.Dropout (0.5),
                                     nn.Linear (32 * 36, 16 * 36),
                                     nn. ReLU (),
                                     nn.Dropout (0.6),
                                     nn.Linear (16 * 36, 2 * 36),
                                     nn.ReLU (),
                                     nn.Dropout (0.6),
                                     nn.Linear (72, 8),
                                     nn.ReLU (),
                                     nn.Dropout (0.6),
                                     nn.Linear (8, 1))

  alexnet.to (device)

  optimizer = torch.optim.Adam (alexnet.classifier.parameters (), lr = 0.001)

  early_stopper = EarlyStopper (patience = 10, min_delta = 0)

  lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau (optimizer, mode = 'min', factor = 0.1, patience = 5)

  model, train_losses, test_losses, train_accs, test_accs, best_epoch, best_acc = train_alexnet (model = alexnet, device = device,
                                                                                               train_dataloader = train_dataloader,
                                                                                               test_dataloader = test_dataloader,
                                                                                              num_epochs = 50, loss_fn = loss_fn,
                                                                                               optimizer = optimizer,
                                                                                               early_stopper = early_stopper,
                                                                                               lr_scheduler = lr_scheduler)

  losses_folds_train.append (train_losses)

  losses_folds_test.append (test_losses)

  accs_folds_train.append (train_accs)

  accs_folds_test.append (test_accs)

  epoch_folds.append (best_epoch)
  model_folds.append (model)
  best_accs_returned_by_the_training_model.append (best_acc)
  best_loss_final, best_accuracy_final = test_alexnet (model = model, device = device, test_dataloader = test_dataloader,
                                                       loss_fn= loss_fn)

  best_accs_returned_by_the_test_model.append (best_accuracy_final)

  acc_folds += best_accuracy_final

print (f'total accuracy is {acc_folds / 5}')
print (f'best epochs are {epoch_folds}')
print (f'best_accuracies {best_accs_returned_by_the_test_model}')

In [ ]:
#Plotting
fig = plt.figure (figsize = (10, 10))
for i in range (n_folds):
  plt.subplot (2, 3, i+1)
  plt.plot (accs_folds_test [i], color = 'green', label = 'test')
  plt.plot (accs_folds_train [i], color = 'red', label = 'train')
  plt.legend ()
plt.savefig ('accs.jpg')
plt.show ()


In [ ]:

fig = plt.figure (figsize = (10, 10))
for i in range (n_folds):
  plt.subplot (2, 3, i+1)
  plt.plot (losses_folds_test [i], color = 'green', label = 'test')
  plt.plot (losses_folds_train [i], color = 'red', label = 'train')
  plt.legend ()
plt.savefig ('losses.jpg')
plt.show ()